# Join and Transformation

Combine the validated university datasets and apply the agreed transformation rules.

In [12]:
from pathlib import Path
import pandas as pd
import re
import unicodedata


project_root = Path("..")
interim_dir = project_root / "data" / "interim"
processed_dir = project_root / "data" / "processed"

In [13]:
kaust = pd.read_csv(
    interim_dir / "KAUST_validated.csv"
)

kfupm = pd.read_csv(
    interim_dir / "KFUPM_validated.csv"
)

print("KAUST:", kaust.shape)
print("KFUPM:", kfupm.shape)

KAUST: (938, 14)
KFUPM: (0, 14)


## Join Specification

- Operation: Row-wise concatenation using `pd.concat`.
- Inputs: `KAUST_validated.csv` and `KFUPM_validated.csv`.
- Join keys: Not applicable; datasets share the same schema.
- Traceability key: `research_id`.
- Expected input: KAUST = 928 rows; KFUPM = 0 rows.
- Expected output before technology filtering: 928 rows.
- Current output after technology filtering: 122 rows.
- Duplicate handling: Duplicate research IDs cause an assertion failure
  and require review; records are not silently removed.
- Cross-source DOI deduplication will be handled during team consolidation.

## Technology Filter Rule

| Rule ID | Description | Input Columns | Output |
|---|---|---|---|
| R4 | Select records containing at least one computing or AI keyword; ignore case, normalize hyphens and spaces, and exclude the phrase “computer vision syndrome” before matching | title, abstract | Filtered records and a separate matching audit |

Missing abstracts are searched by title only. Records without keyword
matches are preserved in `technology_filter/no_keyword_match.csv`.
Selected records are keyword candidates and require sample review.

In [14]:
combined = pd.concat(
    [kaust, kfupm],
    ignore_index=True
)

print("Rows before combine:", len(kaust) + len(kfupm))
print("Rows after combine:", len(combined))
print("Duplicate research IDs:", combined["research_id"].duplicated().sum())

Rows before combine: 938
Rows after combine: 938
Duplicate research IDs: 0


In [15]:
tech_terms = [
    "artificial intelligence", "machine learning", "deep learning",
    "neural network", "neural networks", "multilayer perceptron",
    "random forest", "computer vision", "natural language processing",
    "large language model", "large language models", "generative ai",
    "cybersecurity", "cyber security", "information security",
    "intrusion detection", "cryptography", "blockchain",
    "software engineering", "internet of things", "iot",
    "cloud computing", "edge computing",
    "wireless network", "wireless networks",
    "robotics", "robot", "robots",
    "data mining", "big data", "computer science",
    "ann modeling", "ann modelling"
]

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value)).casefold()
    text = re.sub(r"[-‐‑‒–—−_]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return re.sub(r"\bcomputer vision syndrome\b", " ", text)

patterns = {
    term: re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    for term in tech_terms
}

def find_matches(row):
    title = normalize_text(row["title"])
    abstract = normalize_text(row["abstract"])

    return [
        term
        for term, pattern in patterns.items()
        if pattern.search(title) or pattern.search(abstract)
    ]

# Rebuild from the inputs so rerunning this cell is safe.
before_filter = pd.concat([kaust, kfupm], ignore_index=True)

matched_terms = before_filter.apply(find_matches, axis=1)
keep = matched_terms.map(bool)

combined = before_filter.loc[keep].copy()
excluded_by_technology = before_filter.loc[~keep].copy()

filter_review = before_filter[
    ["research_id", "university", "title"]
].copy()

filter_review["matched_terms"] = matched_terms.map("; ".join)
filter_review["selected"] = keep

print("Before filter:", len(before_filter))
print("Technology candidates:", len(combined))
print("No keyword match:", len(excluded_by_technology))

display(filter_review.loc[keep].head(10))

Before filter: 938
Technology candidates: 123
No keyword match: 815


,research_id,university,title,matched_terms,selected
2,http://hdl.handle.net/10754/691064,KAUST,Semi-universal geo-crack detection by machine ...,machine learning; neural network,True
12,http://hdl.handle.net/10754/673032,KAUST,Learning-based importance sampling via stochas...,neural network,True
23,http://hdl.handle.net/10754/690931,KAUST,Imaging-based intelligent spectrometer on a pl...,deep learning,True
30,http://hdl.handle.net/10754/690918,KAUST,Enhancing DNN models for EEG/ECoG BCI with a N...,machine learning; neural networks,True
31,http://hdl.handle.net/10754/690915,KAUST,Efficient training of spiking neural networks ...,neural networks,True
38,http://hdl.handle.net/10754/690872,KAUST,User-Driven Design and Development of an Under...,robotics,True
42,http://hdl.handle.net/10754/690834,KAUST,Multi-scale geophysical characterization of mi...,machine learning; random forest,True
45,http://hdl.handle.net/10754/690794,KAUST,An artificial neural network-based performance...,neural network,True
60,http://hdl.handle.net/10754/690697,KAUST,Exploiting machine learning models to identify...,machine learning; deep learning,True
63,http://hdl.handle.net/10754/690684,KAUST,"Flexible Oxide Thin Film Transistors, Memristo...",internet of things; iot,True


In [16]:
assert len(combined) + len(excluded_by_technology) == len(before_filter)
assert combined.index.intersection(excluded_by_technology.index).empty
assert combined["research_id"].is_unique

assert "machine learning" in find_matches({
    "title": "A MACHINE-LEARNING approach",
    "abstract": None
})

assert find_matches({
    "title": "Rainfall and pain assessment",
    "abstract": ""
}) == []

assert find_matches({
    "title": "Computer vision syndrome",
    "abstract": None
}) == []

assert "deep learning" in find_matches({
    "title": "Computer vision syndrome detection",
    "abstract": "Using deep learning"
})

assert find_matches({"title": None, "abstract": None}) == []

print("Technology filter tests passed.")

Technology filter tests passed.


In [17]:
combined = combined.drop(
    columns="validation_error",
    errors="ignore"
)

print("Final columns:", len(combined.columns))

Final columns: 13


In [18]:
transformation_rules = pd.DataFrame([
    [
        "R1",
        "Convert publication_year to nullable integer",
        "publication_year",
        "publication_year"
    ],
    [
        "R2",
        "Create a flag showing whether a DOI is available",
        "doi",
        "has_doi"
    ],
    [
        "R3",
        "Calculate the number of words in each abstract",
        "abstract",
        "abstract_word_count"
    ]
], columns=[
    "rule_id",
    "description",
    "input_columns",
    "output_column"
])

transformation_rules

,rule_id,description,input_columns,output_column
0,R1,Convert publication_year to nullable integer,publication_year,publication_year
1,R2,Create a flag showing whether a DOI is available,doi,has_doi
2,R3,Calculate the number of words in each abstract,abstract,abstract_word_count


In [19]:
final_data = combined.copy()

final_data["publication_year"] = pd.to_numeric(
    final_data["publication_year"],
    errors="coerce"
).astype("Int64")

final_data["has_doi"] = (
    final_data["doi"]
    .notna()
    & final_data["doi"].astype("string").str.strip().ne("")
)

final_data["abstract_word_count"] = (
    final_data["abstract"]
    .fillna("")
    .astype("string")
    .str.split()
    .str.len()
)

print("Final rows:", len(final_data))
print("Final columns:", len(final_data.columns))

Final rows: 123
Final columns: 15


In [20]:
assert len(final_data) == len(combined)

assert final_data["research_id"].duplicated().sum() == 0

assert final_data["publication_year"].notna().all()

assert final_data["has_doi"].isin([True, False]).all()

assert (final_data["abstract_word_count"] >= 0).all()

print("All transformation tests passed.")

All transformation tests passed.


In [21]:
processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = processed_dir / "final_rana.csv"

final_data.to_csv(
    output_file,
    index=False
)

saved_final = pd.read_csv(output_file)

print("Saved:", output_file)
print("Saved rows:", len(saved_final))
print("Saved columns:", len(saved_final.columns))

Saved: ..\data\processed\final_rana.csv
Saved rows: 123
Saved columns: 15


In [22]:
filter_dir = processed_dir / "technology_filter"
filter_dir.mkdir(parents=True, exist_ok=True)

filter_review.to_csv(
    filter_dir / "filter_review.csv",
    index=False,
    encoding="utf-8-sig"
)

excluded_by_technology.to_csv(
    filter_dir / "no_keyword_match.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Filter details saved.")

Filter details saved.
